# 🛠️ DoNext 5G — Notebook 3 : Prétraitement (Preprocessing)

**Pipeline :** `NB1_EDA` ✅ → `NB2_Handover_FE` ✅ → `NB3_Preprocessing` → `NB4_Modeling`

---
### 🎯 Objectif
Préparer le dataset feature-engineered pour l'entraînement ML/DL, sans data leakage.

| Étape | Description | Justification |
|-------|-------------|---------------|
| PT-1 | Suppression colonnes inutiles | VarianceThreshold, data leakage |
| PT-2 | Imputation des NaN | Rubin (1976) MCAR/MAR/MNAR |
| PT-3 | Encodage catégoriel | Label Encoding requis XGBoost/LSTM |
| PT-4 | Winsorization IQR | Outliers iperf (Tukey, 1977) |
| PT-5 | Split temporel 70/15/15 | Pas de data leakage temporel |
| PT-6 | Normalisation hybride | Min-Max (LSTM) + Robust (iperf) |
| PT-7 | Sauvegarde finale | Prêt pour NB4_Modeling |

### 📥 Entrée
`FE_output/df_final_fe.parquet` — produit par NB2

### 📤 Sorties
| Fichier | Description |
|---------|-------------|
| `PT_output/df_preprocessed.parquet` | Dataset normalisé final |
| `PT_output/y_train/val/test.npy` | Labels binaires |
| `PT_output/idx_train/val/test.npy` | Index des splits |
| `PT_output/scaler_minmax.pkl` | Scaler Min-Max (inférence) |
| `PT_output/scaler_robust.pkl` | Scaler Robust (inférence) |
| `PT_output/config.json` | Configuration complète |
| `PT_output/mappings.json` | Encodages catégoriels |


In [1]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, gc, json, pickle
import warnings
from sklearn.preprocessing import MinMaxScaler, RobustScaler

warnings.filterwarnings('ignore')

DATASET_ROOT = r'C:\Users\THINKPAD\Desktop\DATASET'  # ← adapter
FE_OUT_DIR   = os.path.join(DATASET_ROOT, 'FE_output')
PT_OUT_DIR   = os.path.join(DATASET_ROOT, 'PT_output')
os.makedirs(PT_OUT_DIR, exist_ok=True)

CONFIGS = {'static': 'session_id', 'mobile': 'device', 'hbahn': 'device'}

print('✅ Setup OK')
assert os.path.exists(FE_OUT_DIR), '❌ FE_output introuvable — exécuter NB2 en premier !'

✅ Setup OK


In [2]:
# ── PT-0 : Chargement ────────────────────────────────────────────────────────
print('='*60)
print('  PT-0 — CHARGEMENT')
print('='*60)

df_final = pd.read_parquet(os.path.join(FE_OUT_DIR, 'df_final_fe.parquet'))
print(f'✅ {len(df_final):,} lignes × {df_final.shape[1]} cols')
print(f'  RAM estimée : {df_final.memory_usage(deep=True).sum()/1e6:.0f} MB')

df = df_final.copy()
del df_final
gc.collect()
print(f'Dataset de travail : {len(df):,} × {df.shape[1]}')

  PT-0 — CHARGEMENT
✅ 12,602,863 lignes × 121 cols
  RAM estimée : 14653 MB
Dataset de travail : 12,602,863 × 121


In [5]:
print('='*60)
print('  PT-1 — SUPPRESSION COLONNES (SAFE + PROGRESS)')
print('='*60)

cols_100_nan = []
cols_unique_1 = []
cols_low_dispo = []

total_cols = len(df.columns)

for i, col in enumerate(df.columns):

    if i % 10 == 0:
        print(f"Progression : {i}/{total_cols}")

    s = df[col]

    try:
        # ⚡ utiliser SAMPLE pour accélérer
        s_sample = s.sample(min(200000, len(s)), random_state=42)

        # 100% NaN
        if s_sample.isna().mean() == 1.0:
            cols_100_nan.append(col)
            continue

        # Valeur unique
        if s_sample.nunique(dropna=True) <= 1:
            cols_unique_1.append(col)
            continue

        # Low dispo
        if s_sample.notna().mean() < 0.05 and col not in ['handover','ho_type']:
            cols_low_dispo.append(col)

    except:
        continue

print("Analyse terminée ✅")

cols_id = [c for c in ['source_file','id','refsig_id','username',
                       'session_start_timestamp','devicename']
           if c in df.columns]

cols_leakage = [c for c in ['cell_index','physical_cellid','earfcn_prev',
                           'mnc_prev','physical_cellid_prev']
                if c in df.columns]

all_drop = list(set(cols_100_nan + cols_unique_1 + cols_id +
                    cols_leakage + cols_low_dispo))

print(f"Colonnes à supprimer : {len(all_drop)}")

df = df.drop(columns=all_drop, errors='ignore')

print(f'✅ Colonnes restantes : {df.shape[1]}')

  PT-1 — SUPPRESSION COLONNES (SAFE + PROGRESS)
Progression : 0/102
Progression : 10/102
Progression : 20/102
Progression : 30/102
Progression : 40/102
Progression : 50/102
Progression : 60/102
Progression : 70/102
Progression : 80/102
Progression : 90/102
Progression : 100/102
Analyse terminée ✅
Colonnes à supprimer : 0
✅ Colonnes restantes : 102


In [6]:
print('='*60)
print('  PT-2 — IMPUTATION ROBUSTE (BIG DATA SAFE)')
print('='*60)

import numpy as np
import pandas as pd

# ── 0. Nettoyage SAFE ───────────────────────────────────────
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].replace(['', '-', ' '], np.nan)

# ── 1. Features binaires ────────────────────────────────────
features_map = {
    'has_gps':'latitude',
    'is_5g':'ss_rsrp',
    'has_iperf':'datarate_mean',
    'has_neigh':'nb_neighbors_mean'
}

for feat, col in features_map.items():
    if col in df.columns:
        df[feat] = df[col].notna().astype('int8')
    else:
        df[feat] = 0

# ── 2. KPI radio ────────────────────────────────────────────
PLAGES_3GPP = {
    'rsrp':(-140,-44),'rsrq':(-19.5,-3),'sinr':(-23,40),
    'rssi':(-120,0),'cqi':(0,15),'tx_power':(-40,23)
}

for col in PLAGES_3GPP:
    if col not in df.columns:
        continue

    lo, hi = PLAGES_3GPP[col]

    df.loc[df[col].notna() & ~df[col].between(lo, hi), col] = np.nan

    med = df[col].median()
    if np.isnan(med):
        med = lo

    df[col] = df[col].fillna(med).clip(lo, hi)

# ── 3. GPS (sans interpolation) ─────────────────────────────
gps_cols = [
    'latitude','longitude','altitude','location_accuracy',
    'velocity','velocity_accuracy','bearing','bearing_accuracy'
]

for col in gps_cols:
    if col in df.columns:
        med = df[col].median()
        df[col] = df[col].fillna(med if not np.isnan(med) else 0)

# ── 4. Numériques ───────────────────────────────────────────
for col in df.columns:
    if np.issubdtype(df[col].dtype, np.number):
        if df[col].isna().any():
            med = df[col].median()
            df[col] = df[col].fillna(med if not np.isnan(med) else 0)

# ── 5. Dates ────────────────────────────────────────────────
for col in df.columns:
    if 'datetime64' in str(df[col].dtype):
        df[col] = df[col].fillna(pd.Timestamp('1970-01-01'))

# ── 6. Catégorielles ────────────────────────────────────────
for col in df.columns:
    if df[col].dtype == 'object':
        if df[col].isna().any():
            mode = df[col].mode()
            df[col] = df[col].fillna(mode[0] if len(mode) > 0 else 'unknown')

# ── 7. Vérification ─────────────────────────────────────────
nan_total = df.isna().sum().sum()

print(f'\nNaN restants : {nan_total}')
print('✅ OK' if nan_total == 0 else '⚠️ NaN restants')

print(f'Shape final : {df.shape}')

  PT-2 — IMPUTATION ROBUSTE (BIG DATA SAFE)

NaN restants : 0
✅ OK
Shape final : (12602863, 106)


In [7]:
# ── PT-3 : Encodage catégoriel ───────────────────────────────────────────────
print('='*60)
print('  PT-3 — ENCODAGE CATÉGORIEL (Label Encoding)')
print('='*60)

# Nettoyage valeurs nulles
df = df.replace(['', ' ', '-', 'NA', 'N/A', 'null'], pd.NA)

# Mappings manuels (logiques métier conservées)
MAPPINGS = {
    'source_folder' : {'static':0,'mobile':1,'hbahn':2},
    'ho_type'       : {'no_handover':0,'intra_freq':1,'inter_freq':2,
                       'inter_RAT_NR':3,'inter_operator':4,
                       'intra_freq_pci':5,'inter_freq_pci':6,'ho_non_type':7},
    'network'       : {'2G':0,'3G':1,'4G':2,'5G NSA':3},
    'network_neighboring' : {'2G':0,'3G':1,'4G':2,'5G':3},
    'MNO'           : {'A':0,'B':1,'C':2},
    'MNO_neighboring': {'A':0,'B':1,'C':2},
    'protocol'      : {'tcp':0,'udp':1},
    'device'        : {'armv7l_RM500Q-GL':0,'armv7l_none':1,
                       'o1s_SM-G991B':2,'r0s_SM-S901B':3,
                       'x86_64_RM500Q-GL':4,'x86_64_RM520N-EU':5},
}

# Liste des colonnes déjà encodées
encoded_cols = []

# Encodage manuel
for col, mapping in MAPPINGS.items():
    if col in df.columns:
        df[f'{col}_enc'] = df[col].map(mapping).fillna(-1).astype(int)
        encoded_cols.append(col)
        print(f'  {col:<25} → {col}_enc  (unique: {df[f"{col}_enc"].nunique()})')

# Encodage spécifique (ex: IP)
if 'server_ip' in df.columns:
    df['server_ip_enc'] = pd.factorize(df['server_ip'])[0]
    encoded_cols.append('server_ip')
    print(f'  {"server_ip":<25} → server_ip_enc')

# Encodage automatique UNIQUEMENT pour colonnes restantes
for col in df.select_dtypes(include='object').columns:
    if col not in encoded_cols:
        df[f'{col}_enc'] = pd.factorize(df[col])[0]
        print(f'  {col:<25} → {col}_enc (auto)')

# 🔥 (Option recommandée) supprimer les colonnes catégorielles originales
df.drop(columns=[col for col in encoded_cols if col in df.columns], inplace=True)

# Sauvegarde des mappings
with open(os.path.join(PT_OUT_DIR, 'mappings.json'), 'w') as f:
    json.dump(MAPPINGS, f, indent=2)

print('\n✅ PT-3 terminé — encodage propre sans duplication')

  PT-3 — ENCODAGE CATÉGORIEL (Label Encoding)
  source_folder             → source_folder_enc  (unique: 3)
  ho_type                   → ho_type_enc  (unique: 8)
  network                   → network_enc  (unique: 4)
  MNO                       → MNO_enc  (unique: 3)
  device                    → device_enc  (unique: 6)

✅ PT-3 terminé — encodage propre sans duplication


In [9]:
# ── PT-4 : Winsorization IQR (features iperf uniquement) ────────────────────
# Note : KPI radio intentionnellement exclus (valeurs extrêmes = situations réelles)
# Référence : Tukey (1977) — Exploratory Data Analysis

print('='*60)
print('  PT-4 — OUTLIERS : IQR Winsorization (iperf only)')
print('='*60)

cols_winsor = [c for c in
    ['pkt_error_mean','datarate_max','tcp_rtt_mean','retrans_mean','datarate_mean']
    if c in df.columns]

print(f'  Colonnes traitées : {cols_winsor}')
for col in cols_winsor:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR    = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_cap  = ((df[col] < lo) | (df[col] > hi)).sum()
    df[col] = df[col].clip(lower=lo, upper=hi)
    print(f'  {col:<25} borne_hi={hi:>10.1f}  cappés={n_cap:,}')

print('\n✅ PT-4 terminé')

  PT-4 — OUTLIERS : IQR Winsorization (iperf only)
  Colonnes traitées : ['pkt_error_mean', 'datarate_max', 'tcp_rtt_mean', 'retrans_mean', 'datarate_mean']
  pkt_error_mean            borne_hi=     194.1  cappés=0
  datarate_max              borne_hi=5404853811.0  cappés=0
  tcp_rtt_mean              borne_hi=   76812.3  cappés=0
  retrans_mean              borne_hi=       3.4  cappés=0
  datarate_mean             borne_hi=87727211.0  cappés=0

✅ PT-4 terminé


In [10]:
# ── PT-5 : Split temporel 70/15/15 par environnement ─────────────────────────
# Principe : ordre temporel TOUJOURS respecté, pas de shuffle
# Référence : Bergmeir & Benítez (2012) — CV for time series

print('='*60)
print('  PT-5 — SPLIT TEMPOREL 70/15/15')
print('='*60)
gc.collect()

y_binary     = df['handover'].values
y_multiclass = df['ho_type_enc'].values if 'ho_type_enc' in df.columns else None

cols_X = [c for c in df.columns if c not in ['handover','ho_type_enc']]
print(f'  Features (X) : {len(cols_X)}')

idx_trains, idx_vals, idx_tests = [], [], []
y_trains,   y_vals,   y_tests   = [], [], []

print(f'  {"Env":<10} {"Total":>10} {"Train":>10} {"Val":>10} {"Test":>10} {"HO%":>8}')
print('  ' + '─'*58)

for env_code, env_name in [(0,'static'),(1,'mobile'),(2,'hbahn')]:
    idx_env = df.index[df['source_folder_enc'] == env_code].tolist()
    n       = len(idx_env)
    if n == 0:
        print(f'  {env_name:<10} — aucune donnée')
        continue
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)

    idx_trains.append(idx_env[:n_train])
    idx_vals.append(  idx_env[n_train:n_train+n_val])
    idx_tests.append( idx_env[n_train+n_val:])

    y_env = y_binary[idx_env]
    y_trains.append(y_env[:n_train])
    y_vals.append(  y_env[n_train:n_train+n_val])
    y_tests.append( y_env[n_train+n_val:])

    print(f'  {env_name:<10} {n:>10,} {n_train:>10,} {n_val:>10,} '
          f'{n-n_train-n_val:>10,} {y_env[:n_train].mean()*100:>7.2f}%')
    gc.collect()

idx_train_all = sum(idx_trains, [])
idx_val_all   = sum(idx_vals, [])
idx_test_all  = sum(idx_tests, [])
y_train = np.concatenate(y_trains)
y_val   = np.concatenate(y_vals)
y_test  = np.concatenate(y_tests)

print(f'\n  Train : {len(idx_train_all):,}  |  Val : {len(idx_val_all):,}  |  Test : {len(idx_test_all):,}')
print('\n✅ PT-5 terminé')

  PT-5 — SPLIT TEMPOREL 70/15/15
  Features (X) : 104
  Env             Total      Train        Val       Test      HO%
  ──────────────────────────────────────────────────────────
  static      9,725,885  6,808,119  1,458,882  1,458,884    3.13%
  mobile      2,382,602  1,667,821    357,390    357,391    5.20%
  hbahn         494,376    346,063     74,156     74,157   17.31%

  Train : 8,822,003  |  Val : 1,890,428  |  Test : 1,890,432

✅ PT-5 terminé


In [11]:
# ── PT-6 : Normalisation hybride Min-Max + RobustScaler ──────────────────────
# Min-Max  → KPI signal, GPS, fenêtre T-k (range [0,1] requis pour LSTM)
# Robust   → features iperf/latency (distribution très asymétrique)
# Référence : Géron (2022) — Hands-On ML with Scikit-Learn, Keras & TF (ch.2)

print('='*60)
print('  PT-6 — NORMALISATION HYBRIDE')
print('='*60)

def numeric_only(df, cols):
    return [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]

cols_no_norm = [c for c in
    ['source_folder_enc','network_enc','MNO_enc','device_enc',
     'has_gps','is_5g','has_iperf','has_neigh',
     'session_id','passive_id','timestamp','timestamp_day',
     'week_of_year','day_of_week']
    if c in cols_X]

cols_robust  = numeric_only(df, [c for c in
    ['datarate_mean','datarate_max','tcp_rtt_mean','retrans_mean',
     'pkt_error_mean','packet_loss_mean','mean_latency',
     'mean_dev_latency','min_latency','max_latency',
     'nb_neighbors_pid','nb_neighbors_mean','nb_neighbors_max']
    if c in cols_X and c not in cols_no_norm])

cols_minmax  = numeric_only(df, [c for c in cols_X
                                  if c not in cols_no_norm and c not in cols_robust])

X_train_ref = df.loc[idx_train_all, :]

scaler_mm = MinMaxScaler()
if cols_minmax:
    scaler_mm.fit(X_train_ref[cols_minmax])
    df[cols_minmax] = scaler_mm.transform(df[cols_minmax])

scaler_rb = RobustScaler()
if cols_robust:
    scaler_rb.fit(X_train_ref[cols_robust])
    df[cols_robust] = scaler_rb.transform(df[cols_robust])

print(f'  Exclus     : {len(cols_no_norm)}')
print(f'  Min-Max    : {len(cols_minmax)}')
print(f'  Robust     : {len(cols_robust)}')
print('\n✅ PT-6 terminé')

  PT-6 — NORMALISATION HYBRIDE
  Exclus     : 14
  Min-Max    : 79
  Robust     : 11

✅ PT-6 terminé


In [12]:
# ── PT-7 : Sauvegarde dataset final ──────────────────────────────────────────
print('='*60)
print('  PT-7 — SAUVEGARDE FINALE')
print('='*60)

np.save(os.path.join(PT_OUT_DIR, 'idx_train.npy'), np.array(idx_train_all))
np.save(os.path.join(PT_OUT_DIR, 'idx_val.npy'),   np.array(idx_val_all))
np.save(os.path.join(PT_OUT_DIR, 'idx_test.npy'),  np.array(idx_test_all))
np.save(os.path.join(PT_OUT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(PT_OUT_DIR, 'y_val.npy'),   y_val)
np.save(os.path.join(PT_OUT_DIR, 'y_test.npy'),  y_test)

path_final = os.path.join(PT_OUT_DIR, 'df_preprocessed.parquet')
df.to_parquet(path_final, index=False, compression='snappy')

with open(os.path.join(PT_OUT_DIR, 'scaler_minmax.pkl'), 'wb') as f:
    pickle.dump(scaler_mm, f)
with open(os.path.join(PT_OUT_DIR, 'scaler_robust.pkl'), 'wb') as f:
    pickle.dump(scaler_rb, f)

config = {
    'cols_X': cols_X, 'cols_minmax': cols_minmax,
    'cols_robust': cols_robust, 'cols_no_norm': cols_no_norm,
    'n_train': len(idx_train_all), 'n_val': len(idx_val_all),
    'n_test': len(idx_test_all),
    'ho_rate_train': float(y_train.mean()),
}
with open(os.path.join(PT_OUT_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

size = os.path.getsize(path_final) / 1e6
print(f'  df_preprocessed.parquet : {len(df):,} lignes × {df.shape[1]} cols → {size:.1f} MB')
print(f'  y_train {y_train.shape}  HO={y_train.mean()*100:.2f}%')
print(f'  y_val   {y_val.shape}    HO={y_val.mean()*100:.2f}%')
print(f'  y_test  {y_test.shape}   HO={y_test.mean()*100:.2f}%')
print('\n✅ PRÉTRAITEMENT COMPLET → NB4_Modeling.ipynb')

  PT-7 — SAUVEGARDE FINALE
  df_preprocessed.parquet : 12,602,863 lignes × 106 cols → 412.7 MB
  y_train (8822003,)  HO=4.08%
  y_val   (1890428,)    HO=9.73%
  y_test  (1890432,)   HO=22.83%

✅ PRÉTRAITEMENT COMPLET → NB4_Modeling.ipynb


---
## Bilan Notebook 3

### ✅ Réalisé
Imputation propre (3GPP + MCAR/MAR), encodage, Winsorization, split temporel sans data leakage, normalisation hybride.

### 📚 Références
- Rubin, D.B. (1976) — *Inference and missing data*, Biometrika
- Tukey, J.W. (1977) — *Exploratory Data Analysis*
- Bergmeir & Benítez (2012) — *On the use of CV for time series*, Inf. Sciences
- Géron, A. (2022) — *Hands-On ML* (3rd ed.), O'Reilly

### 🚀 Prochaine étape
→ **NB4_Modeling.ipynb**
